<a href="https://colab.research.google.com/github/usama488/bioinformatics-analysis/blob/main/Research_Paper_AI_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Research Paper AI Chatbot

# Project 7 Bioinformatics

Is notebook mein hum ek Research Paper Q&A Chatbot banain gay jo kisi bhi scientific paper (PDF/text) ko parh kar uske content ke bare mein sawalon ka jawab de sakta hai RAG (Retrieval-Augmented Generation) approach use kar ke.

# **Pipeline:**
# 1. Paper load karna (PDF ya text)
# 2. Text chunking (paragraphs mein todna)
# 3. TF-IDF embeddings + semantic search (retrieval)
# 4. Extractive Q&A (offline, bina API key ke)
# 5. Optional: LLM API integration (OpenAI/Anthropic) generative answers ke liye
# 6. Interactive Plotly visualizations (chunk similarity, keyword importance)
# 7. Runtime chatbot interface — apna sawal type karein aur turant jawab paayein


## 1. Setup & Imports

In [ ]:
# !pip install -q scikit-learn pandas numpy plotly ipywidgets pdfplumber

import numpy as np
import pandas as pd
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import plotly.express as px
import plotly.graph_objects as go

import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

np.random.seed(42)


## 2. Load Research Paper


**Option A — Apna PDF upload karein (Colab mein):**
```python
from google.colab import files
uploaded = files.upload()
import pdfplumber
with pdfplumber.open(list(uploaded.keys())[0]) as pdf:
    paper_text = "\n".join(page.extract_text() or "" for page in pdf.pages)
```

**Option B — Neeche di gayi sample paper text use karein** (demo ke liye — abstract-style bioinformatics paper), taake bina upload ke bhi pura pipeline turant chal sake.


In [1]:
paper_text = """
Title: Machine Learning Approaches for Cancer Classification Using Gene Expression Data

Abstract: Cancer classification based on gene expression profiling has become an important area of research in bioinformatics. In this study, we evaluate several machine learning algorithms, including Random Forest, Support Vector Machines, and Logistic Regression, for their ability to classify tumor samples as malignant or benign using RNA-sequencing data. Our dataset consists of 569 patient samples with 30 quantitative features derived from digitized images of fine needle aspirate biopsies.

Introduction: Breast cancer remains one of the most commonly diagnosed cancers among women worldwide. Early and accurate diagnosis significantly improves patient survival rates. Traditional diagnostic methods rely on histopathological examination, which can be subjective and time-consuming. Machine learning offers a data-driven alternative that can assist pathologists by providing rapid, consistent predictions based on quantitative cell nuclei measurements.

Methods: We preprocessed the dataset by removing samples with missing values and standardizing all numerical features using z-score normalization. The dataset was split into training (80%) and testing (20%) sets using stratified sampling to preserve class balance. Four classifiers were trained: Logistic Regression, Random Forest with 300 estimators, a Support Vector Machine with a radial basis function kernel, and Gradient Boosting. Hyperparameters were tuned using 5-fold cross-validation with grid search over key parameters such as regularization strength and tree depth.

Results: The Random Forest classifier achieved the highest performance with an accuracy of 96.5% and an ROC-AUC of 0.99 on the held-out test set. Feature importance analysis revealed that mean concave points, worst area, and worst concavity were the most predictive features for distinguishing malignant from benign tumors. The Support Vector Machine achieved comparable performance with an accuracy of 95.6%. Gradient Boosting slightly underperformed relative to Random Forest, achieving 94.8% accuracy, likely due to overfitting on the smaller training set.

Discussion: Our results demonstrate that ensemble-based methods such as Random Forest are well-suited for high-dimensional gene expression and biopsy feature data, as they naturally handle feature interactions and provide interpretable feature importance scores. The high recall for the malignant class is particularly important in a clinical context, as false negatives could delay treatment. Future work should explore deep learning architectures and validate these findings on independent, multi-institutional datasets to assess generalizability across different populations and imaging protocols.

Conclusion: Machine learning models, particularly Random Forest, can achieve high diagnostic accuracy for breast cancer classification using quantitative biopsy features. These tools have the potential to serve as decision-support systems for pathologists, though clinical deployment requires further validation, regulatory approval, and integration into existing diagnostic workflows.

Keywords: cancer classification, machine learning, gene expression, random forest, breast cancer, bioinformatics
""".strip()

print(f"Paper loaded: {len(paper_text)} characters, {len(paper_text.split())} words")


Paper loaded: 3293 characters, 432 words


## 3. Text Chunking

In [ ]:
def chunk_text(text, min_len=40):
    # Split by section-like paragraphs (blank-line or heading pattern), then by sentences if needed
    raw_chunks = [c.strip() for c in re.split(r'\n\s*\n', text) if c.strip()]
    chunks = []
    for c in raw_chunks:
        if len(c) > 600:
            sentences = re.split(r'(?<=[.!?]) +', c)
            buf = ""
            for s in sentences:
                if len(buf) + len(s) < 400:
                    buf += " " + s
                else:
                    if len(buf.strip()) > min_len:
                        chunks.append(buf.strip())
                    buf = s
            if len(buf.strip()) > min_len:
                chunks.append(buf.strip())
        elif len(c) > min_len:
            chunks.append(c)
    return chunks

chunks = chunk_text(paper_text)
chunk_df = pd.DataFrame({"chunk_id": range(len(chunks)), "text": chunks,
                          "section": [c.split(":")[0] if ":" in c[:30] else f"Chunk {i}" for i, c in enumerate(chunks)]})

print(f"Total chunks: {len(chunks)}")
chunk_df[["chunk_id", "section"]]


## 4. Build TF-IDF Retrieval Index

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", max_features=500, ngram_range=(1, 2))
chunk_vectors = vectorizer.fit_transform(chunk_df["text"])

print(f"TF-IDF matrix shape: {chunk_vectors.shape}  (chunks x vocabulary)")

feature_names = vectorizer.get_feature_names_out()
mean_tfidf = np.asarray(chunk_vectors.mean(axis=0)).flatten()
top_terms = pd.Series(mean_tfidf, index=feature_names).sort_values(ascending=False).head(20)

fig = px.bar(top_terms, orientation='h', title="Top 20 Important Terms in the Paper (TF-IDF)",
             labels={"value": "Avg. TF-IDF Score", "index": "Term"}, template="plotly_white",
             color=top_terms.values, color_continuous_scale="Viridis")
fig.update_layout(height=550, showlegend=False, yaxis={'categoryorder': 'total ascending'})
fig.show()


In [ ]:
sim_matrix = cosine_similarity(chunk_vectors)

fig = px.imshow(sim_matrix, color_continuous_scale="Viridis", aspect="auto",
                 title="Chunk-to-Chunk Similarity Matrix (Semantic Structure of the Paper)",
                 labels=dict(color="Cosine Similarity"),
                 x=[f"C{i}" for i in range(len(chunks))], y=[f"C{i}" for i in range(len(chunks))])
fig.update_layout(height=550)
fig.show()


## 5. Retrieval Function — Find Relevant Chunks for a Question

In [ ]:
def retrieve(question, top_k=3):
    q_vector = vectorizer.transform([question])
    sims = cosine_similarity(q_vector, chunk_vectors).flatten()
    top_idx = sims.argsort()[::-1][:top_k]
    return [(chunk_df.iloc[i]["text"], sims[i], i) for i in top_idx if sims[i] > 0]

def extractive_answer(question, top_k=3):
    results = retrieve(question, top_k)
    if not results:
        return "Is sawal ka jawab paper mein nahi mila. Alag tarah se poochh kar dekhein.", []
    best_chunk, best_score, chunk_id = results[0]
    sentences = re.split(r'(?<=[.!?]) +', best_chunk)

    q_words = set(re.findall(r'\w+', question.lower()))
    scored_sentences = []
    for s in sentences:
        s_words = set(re.findall(r'\w+', s.lower()))
        overlap = len(q_words & s_words)
        scored_sentences.append((s, overlap))
    scored_sentences.sort(key=lambda x: x[1], reverse=True)
    answer = " ".join([s for s, _ in scored_sentences[:2]])
    return answer, results

# Quick test
test_q = "What was the accuracy of the Random Forest model?"
answer, sources = extractive_answer(test_q)
print(f"Q: {test_q}\nA: {answer}\n\nTop source similarity: {sources[0][1]:.3f}")


## 6. (Optional) Connect a Real LLM for Generative Answers

Agar aap chahte hain ke chatbot **generative, natural-language** answers de (sirf extractive nahi), to retrieved chunks ko context ke tor par ek LLM API ko pass karein. Neeche OpenAI-style example hai — Anthropic ya kisi bhi LLM provider ke sath easily adapt ho sakta hai.

```python
 from openai import OpenAI
client = OpenAI(api_key="YOUR_API_KEY")

## def llm_answer(question, top_k=3):
#     _, sources = extractive_answer(question, top_k)
#     context = "\n\n".join([s[0] for s in sources])
#     prompt = f"""Answer the question using ONLY the context below.
# Context:
# {context}
#
# Question: {question}
# Answer:"""
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}]
#     )
#     return response.choices[0].message.content
```

Is notebook ka chatbot (neeche) **bina API key ke offline extractive mode** mein chalta hai — API integrate karna optional hai.


## 7. 🤖 Runtime Chatbot — Apna Sawal Poochhein

Neeche chat interface hai — paper ke bare mein koi bhi sawal type karein, chatbot relevant section dhoond kar jawab dega, source chunk ke sath.


In [ ]:
chat_history = []

question_box = widgets.Text(
    placeholder="Paper ke bare mein sawal type karein... (e.g. What methods were used?)",
    layout=widgets.Layout(width='600px', height='38px')
)
ask_btn = widgets.Button(description="💬 Poochein", button_style='success', layout=widgets.Layout(width='140px', height='38px'))
clear_btn = widgets.Button(description="🗑️ Clear Chat", layout=widgets.Layout(width='140px', height='38px'))
chat_out = widgets.Output()

def render_chat():
    with chat_out:
        clear_output()
        if not chat_history:
            display(HTML('<div style="color:#888; font-family:sans-serif; padding:10px;">Chat khali hai — upar sawal likh kar shuru karein.</div>'))
            return
        html = '<div style="font-family:sans-serif; max-height:420px; overflow-y:auto; padding:6px;">'
        for turn in chat_history:
            html += f"""
            <div style="display:flex; justify-content:flex-end; margin:8px 0;">
                <div style="background:#2E86AB; color:white; padding:10px 16px; border-radius:16px 16px 2px 16px; max-width:70%; font-size:14px;">
                    {turn['question']}
                </div>
            </div>
            <div style="display:flex; justify-content:flex-start; margin:8px 0;">
                <div style="background:#f0f0f0; color:#222; padding:10px 16px; border-radius:16px 16px 16px 2px; max-width:75%; font-size:14px;">
                    🤖 {turn['answer']}
                    <div style="font-size:11px; color:#888; margin-top:6px;">Source similarity: {turn['score']:.2f} | Chunk #{turn['chunk_id']}</div>
                </div>
            </div>
            """
        html += '</div>'
        display(HTML(html))

def on_ask(b):
    q = question_box.value.strip()
    if not q:
        return
    answer, sources = extractive_answer(q)
    score = sources[0][1] if sources else 0.0
    chunk_id = sources[0][2] if sources else -1
    chat_history.append({"question": q, "answer": answer, "score": score, "chunk_id": chunk_id})
    question_box.value = ""
    render_chat()

def on_clear(b):
    chat_history.clear()
    render_chat()

ask_btn.on_click(on_ask)
clear_btn.on_click(on_clear)
question_box.on_submit(on_ask)

display(widgets.HTML("<b style='font-size:16px;'>🤖 Research Paper Chatbot</b>"))
display(widgets.HBox([question_box, ask_btn, clear_btn]))
display(chat_out)
render_chat()
